In [1]:
import sys
import os
project_root = "/Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow"
sys.path.append(sys.path.append(project_root))
path_root = "/Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow"

In [2]:
import numpy as np
import pandas as pd
import joblib
import torch

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel,ConstantKernel,Matern
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from Data_Extraction.DataPreprocessor import DataPreprocessor
from Data_Extraction.database_manager import DatabaseManager

from torch.utils.data import DataLoader, TensorDataset
from Models.VAE.VAE_Model import VAE

from Data_Extraction.AdaptiveTransferKernel import AdaptiveTransferKernel
from sklearn.manifold import TSNE
from Notebooks.DescriptorEngineer import AlloyDescriptorCalculator
from Notebooks.DescriptorEngineer import ElementPropertyLoader

In [18]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Ternary_round1.db")

tables = db.list_tables()
df_optical = db.table_dataframe('Optical_properties')
df_comp = db.table_dataframe('compositions')
print (tables)
db.close()

# --- Merge and filter wavelength robustly ---
df_merged = df_optical.merge(df_comp, on="ID", how="inner")

# safer wavelength filter
tern_1550 = df_merged[np.isclose(df_merged["wavelength_nm"].astype(float), 1552.0)]

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Ternary_round1.db
['compositions', 'Thickness', 'Optical_properties']
Database connection closed


In [4]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Synthetic_data_EMA.db")
tables = db.list_tables()
comp_all_space = db.table_dataframe(table_name='compositions')
db.close()

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Synthetic_data_EMA.db
Database connection closed


Intrinsic Parameters
1. Average Atomic Radius
2. Atomic Radius mismatch
3. Electronegativity difference
4. Mixing entropy
5. Average valence electron concentration

In [5]:
atomic_radius = pd.read_pickle(path_root+"/Data_Extraction/Atomic_radius.pkl")
electronegativity = pd.read_pickle(path_root+"/Data_Extraction/electronegativity.pkl")
valence_electrons = pd.read_pickle(path_root+"/Data_Extraction/Valence_electrons.pkl")

atomic_radius = atomic_radius.set_index("symbol")
electronegativity = electronegativity.set_index("Symbol")
valence_electrons = valence_electrons.set_index("symbol")

In [7]:

atomic_radius = atomic_radius.rename(columns={"Metallic": "atomic_radius"})
valence_electrons = valence_electrons.rename(columns={"valence": "valence_electrons"})

elem_props = (
    atomic_radius
    .join(electronegativity)
    .join(valence_electrons)
)

Properties functions

In [7]:
def get_elem_cols(df_comp: pd.DataFrame, elem_props: pd.DataFrame):
    return df_comp.columns.intersection(elem_props.index)

def ave_property(df_comp: pd.DataFrame, elem_props: pd.DataFrame, prop: str, name: str):
    df = df_comp.copy()
    elem_cols = get_elem_cols(df, elem_props)

    C = df[elem_cols].apply(pd.to_numeric, errors="coerce")

    P = pd.to_numeric(elem_props.loc[elem_cols, prop], errors="coerce")
    df[name] = C.mul(P, axis=1).sum(axis=1)
    return df

def ar_mismatch(df_comp: pd.DataFrame,
                elem_props: pd.DataFrame,
                r_col: str,
                name: str = "delta_r",
                r_ave_col: str = "r_ave") -> pd.DataFrame:
    df = df_comp.copy()
    elem_cols = get_elem_cols(df, elem_props)

    C = df.loc[:, elem_cols].apply(pd.to_numeric)

    r = pd.to_numeric(elem_props.loc[elem_cols, r_col])

    # r_ave
    if r_ave_col not in df.columns:
        df[r_ave_col] = C.mul(r, axis=1).sum(axis=1)

    r_ave = pd.to_numeric(df[r_ave_col]).to_numpy()  # (n_rows,)
    r_vec = r.to_numpy()                                              # (n_elem,)

    ratio = r_vec[None, :] / r_ave[:, None]

    # delta^2 = sum_i C_i (1 - r_i/r_ave)^2
    delta2 = (C.to_numpy() * (1 - ratio) ** 2).sum(axis=1)

    df[name] = np.sqrt(delta2)
    return df

def en_diff(df_comp: pd.DataFrame, elem_props: pd.DataFrame, en_col: str,
            name="delta_EN"):
    df = df_comp.copy()
    elem_cols = get_elem_cols(df, elem_props)

    C = df[elem_cols].apply(pd.to_numeric, errors="coerce")

    EN = pd.to_numeric(elem_props.loc[elem_cols, en_col], errors="coerce")

    EN_avg = C.mul(EN, axis=1).sum(axis=1)
    delta2 = C.mul((EN_avg.values[:, None] - EN.values[None, :]) ** 2, axis=1).sum(axis=1)
    df[name] = np.sqrt(delta2)
    return df

def mix_entropy(df_comp: pd.DataFrame, elem_props: pd.DataFrame,
                name="S_mix", R=8.314):
    df = df_comp.copy()
    elem_cols = get_elem_cols(df, elem_props)

    C = df[elem_cols].apply(pd.to_numeric, errors="coerce")

    C_safe = C.replace(0, np.nan)
    df[name] = (-R * (C_safe * np.log(C_safe)).sum(axis=1)).fillna(0)
    return df

def add_all_descriptors(df_comp: pd.DataFrame, elem_props: pd.DataFrame) -> pd.DataFrame:
    df = df_comp.copy()

    df = ave_property(df, elem_props, prop="atomic_radius", name="r") # Average Atomic Radius
    df = ar_mismatch(df, elem_props, r_col="atomic_radius", name="del_r", r_ave_col="r_ave") # Atomic Radius Mismatch
    df = en_diff(df, elem_props, en_col="electronegativity", name="del_EN") # Electronegativity difference
    df = mix_entropy(df, elem_props, name="S", R=8.314) # Mixing Entropy [J/mol-K]
    df = ave_property(df, elem_props, prop="valence_electrons", name="VEC") # Average Valence electron concentration

    return df


In [13]:
from pathlib import Path
path_properties = path_root + '/Data_Extraction'
path_properties

'/Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction'

In [14]:
from pathlib import Path

#path_properties = Path(path_root) / "Data_Extraction"

loader = ElementPropertyLoader(path_properties)
elem_props = loader.load()

In [15]:
calc = AlloyDescriptorCalculator(elem_props=elem_props)

df_ternary_1f_properties = calc.add_all_descriptors(df_compositions)
# or individually:
#df1 = calc.average_property(df_comp, prop="atomic_radius", out_col="r")

In [16]:
df_ternary_1f_properties

,ID,Cu,Ni,Al,r,r_ave,del_r,del_EN,S,VEC
0,1,0.950,0.050,0.000,127.800,127.800,0.006821,0.002179,1.650456,1.050
1,2,0.950,0.000,0.050,128.750,128.750,0.025392,0.063204,1.650456,1.100
2,3,0.000,0.950,0.050,124.950,124.950,0.033141,0.065383,1.650456,2.050
3,4,0.050,0.950,0.000,124.200,124.200,0.007019,0.002179,1.650456,1.950
4,5,0.050,0.000,0.950,142.250,142.250,0.022982,0.063204,1.650456,2.900
5,6,0.000,0.050,0.950,142.050,142.050,0.029151,0.065383,1.650456,2.950
6,7,0.330,0.330,0.330,130.350,130.350,0.063232,0.139589,9.125247,1.980
7,8,0.950,0.025,0.025,128.275,128.275,0.019015,0.045343,1.938597,1.075
8,9,0.475,0.500,0.025,126.375,126.375,0.026226,0.046340,6.588054,1.550
9,10,0.500,0.500,0.000,126.000,126.000,0.015873,0.005000,5.762826,1.500


In [8]:
df_ternary_1f_properties = add_all_descriptors(df_compositions, elem_props)
comp_all_properties = add_all_descriptors(comp_all_space, elem_props)

In [9]:
comp_all_properties.to_csv(
    'cualni_compositions.csv',
    index=False
)

In [22]:
Data_AFLOW = pd.read_pickle(path_root+"/Data_Extraction/DataFrame_composition_AFLOW.pkl")
elements = Data_AFLOW.columns.to_numpy()

valence_electrons = pd.read_pickle(path_root+"/Data_Extraction/Valence_electrons.pkl")
value_map = pd.to_numeric(valence_electrons.set_index("symbol")["valence"],errors="coerce")

thermal_AFLOW = pd.read_pickle(path_root+"/Data_Extraction/DataFrame_AFLOW_Data.pkl")
thermal_AFLOW = thermal_AFLOW[['thermal_conductivity']]

In [23]:
elem_cols = Data_AFLOW.columns.intersection(value_map.index)
X = Data_AFLOW.loc[:, elem_cols].apply(pd.to_numeric, errors="coerce")
Data_AFLOW["VEC"] = X.mul(value_map.loc[elem_cols], axis=1).sum(axis=1)

In [24]:
prope_columns = ['ID','e1','e2']

df_ternary_expand = pd.concat(
    [tern_1_1550[prope_columns],
     tern_1_1550.reindex(columns=elements, fill_value=0)*1/100  
    ], axis=1
)
df_ternary_expand = df_ternary_expand.reset_index(drop=True)

In [25]:
elem_cols = df_ternary_expand.columns.intersection(value_map.index)
X = df_ternary_expand.loc[:, elem_cols].apply(pd.to_numeric, errors="coerce")
df_ternary_expand["VEC"] = X.mul(value_map.loc[elem_cols], axis=1).sum(axis=1)

In [26]:
DP = DataPreprocessor()
X = df_ternary_expand.drop(columns=['ID','e1','e2'])
y = df_ternary_expand['e2']

X_train, X_test, y_train,y_test = DP.split_training(X,y)

In [27]:
y_exp_train = y_train.to_numpy()
y_exp_test = y_test.to_numpy()
y_therma = thermal_AFLOW.to_numpy(dtype=float)
y_therma = y_therma.reshape(-1)

In [28]:
# Source and target variables
X_s = Data_AFLOW.to_numpy(dtype=float)
X_t_train = X_train.to_numpy(dtype=float)

Xs_aug = np.c_[X_s, np.zeros((len(X_s), 1))]
Xt_aug = np.c_[X_t_train, np.ones((len(X_t_train), 1))]

X_t_test = X_test.to_numpy()
Xt_test_aug = np.c_[X_t_test, np.ones((len(X_t_test), 1))]

X_train_gp = np.vstack([Xs_aug, Xt_aug])
y_train_gp = np.concatenate([y_therma, y_exp_train])

In [60]:
# Fit GP regressor

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor

kernel = AdaptiveTransferKernel(
    kernel=1.0 * Matern(length_scale=1.0, nu=2.5),
    lamb=2.0,
    lamb_bounds=(1.0, 3.0),
    different_noises=False,
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-5,            # jitter for numerical stability
    normalize_y=True,      # y standardization inside sklearn
    n_restarts_optimizer=3 # increase later (5-10) if slow/unstable
)

In [61]:
gpr.fit(X_train_gp,y_train_gp)

/Users/linarojas/opt/anaconda3/envs/Python310/lib/python3.10/site-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


GaussianProcessRegressor(alpha=1e-05,
                         kernel=AdaptiveTransferKernel(0, 0, 0.693),
                         n_restarts_optimizer=3, normalize_y=True)

In [62]:

yhat_tr, y_std = gpr.predict(Xt_test_aug, return_std=True)

mae  = mean_absolute_error(yhat_tr, y_exp_test)
rmse = np.sqrt(mean_squared_error(yhat_tr, y_exp_test))
r2   = r2_score(yhat_tr, y_exp_test)

print(f"MAE : {mae:.4g}")
print(f"RMSE: {rmse:.4g}")
print(f"R^2 : {r2:.4g}")

MAE : 6.543
RMSE: 7.84
R^2 : 0.5283


In [29]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Synthetic_data_EMA.db")
tables = db.list_tables()
comp_all_space = db.table_dataframe(table_name='compositions')
db.close()

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Synthetic_data_EMA.db
Database connection closed


Experimental data GP

In [10]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Ternary_round1.db")

tables = db.list_tables()
df_optical = db.table_dataframe('Optical_properties')
df_compositions = db.table_dataframe('compositions')
print (tables)
db.close()

df_tern_1_rou = pd.merge(df_optical,df_compositions, on='ID')
df_tern_1_rou[["Cu","Ni","Al"]] = df_tern_1_rou[["Cu","Ni","Al"]]
tern_1_1550 = df_tern_1_rou[df_tern_1_rou["wavelength_nm"] == 1552]

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Ternary_round1.db
['compositions', 'Thickness', 'Optical_properties']
Database connection closed


In [35]:
tern_1_1550_comp = tern_1_1550[["Cu","Ni","Al"]].reset_index(drop=True)
tern_1_1550_properties = add_all_descriptors(tern_1_1550_comp, elem_props)

In [96]:
DP = DataPreprocessor()
X = tern_1_1550_properties
y = tern_1_1550['e2']

X_train_split, X_test_split, y_train,y_test = DP.split_training(X,y)

X_train = X_train_split.drop(columns=['r','r_ave','S'])
X_test = X_test_split.drop(columns=['r','r_ave','S'])

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

X_train = np.asarray(X_train, dtype=float)
y_train = np.asarray(y_train, dtype=float).ravel()

kernel = (
    C(1.0, (1e-3, 1e3)) *
    Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
)

gpr_nt = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,   
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gpr_nt.fit(X_train, y_train)
print("Learned kernel:", gpr_nt.kernel_)

Learned kernel: 1.06**2 * Matern(length_scale=0.675, nu=2.5) + WhiteKernel(noise_level=0.248)


In [100]:
X_test = np.asarray(X_test, dtype=float)
y_pred, y_std_nt = gpr_nt.predict(X_test, return_std=True)

mae  = mean_absolute_error(y_pred, y_test)
rmse = np.sqrt(mean_squared_error(y_pred, y_test))
r2   = r2_score(y_pred, y_test)

print(f"MAE : {mae:.4g}")
print(f"RMSE: {rmse:.4g}")
print(f"R^2 : {r2:.4g}")

MAE : 7.241
RMSE: 7.832
R^2 : 0.7045


In [ ]:
x_all_space = np.asarray(comp_all_properties[['Cu','Ni','Al','VEC','del_r','del_EN']], dtype=float)
y_pred_exp, y_std_nt_exp = gpr_nt.predict(x_all_space, return_std=True)

np.save(path_root+"/Data_Extraction/Results/GP-Experiments/e2_predicted_GP.npy",y_pred_exp)

Just Simulated data

In [75]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Pure_elements.db")

tables = db.list_tables()
df_optical_pe = db.table_dataframe('Optical_properties')
df_compositions_pe = db.table_dataframe('compositions')
print (tables)
db.close()

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Pure_elements.db
['compositions', 'sqlite_sequence', 'optical_properties', 'lorentz']
Database connection closed


In [76]:
df_optical_pe = df_optical_pe[df_optical_pe["wavelength_nm"] == 1552]
df_pure_elements = pd.merge(df_compositions_pe,df_optical_pe, on='ID')

In [77]:
prope_columns = ['ID','e1','e2']
df_pe_expanded = DP.expan_comp_df(df_pure_elements,Data_AFLOW,prope_columns=prope_columns)
df_pr_DR = DP.composition_DR_ele_P(elect_data_TSNE,df_pe_expanded)

X_pe = df_pr_DR
Y_pe = df_pe_expanded['e2']

In [78]:
y_pe_train = Y_pe.to_numpy()
y_therma = thermal_AFLOW.to_numpy(dtype=float)
y_therma = y_therma.reshape(-1)

In [79]:
# Source and target variables
X_s = df_thermal.to_numpy(dtype=float)
X_pe_train = X_pe.to_numpy(dtype=float)

Xs_aug = np.c_[X_s, np.zeros((len(X_s), 1))]
Xt_pe_aug = np.c_[X_pe_train, np.ones((len(X_pe_train), 1))]

X_train_gp_pe = np.vstack([Xs_aug, Xt_pe_aug])
y_train_gp_pe = np.concatenate([y_therma, y_pe_train])

In [80]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor

kernel = AdaptiveTransferKernel(
    kernel=1.0 * Matern(length_scale=1.0, nu=2.5),
    lamb=2.0,
    lamb_bounds=(1.0, 3.0),
    different_noises=False,
)

gpr_pe = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-5,            # jitter for numerical stability
    normalize_y=True,      # y standardization inside sklearn
    n_restarts_optimizer=3 # increase later (5-10) if slow/unstable
)

In [81]:
gpr_pe.fit(X_train_gp_pe,y_train_gp_pe)

GaussianProcessRegressor(alpha=1e-05,
                         kernel=AdaptiveTransferKernel(0, 0, 0.693),
                         n_restarts_optimizer=3, normalize_y=True)

In [86]:
X_t_test = df_exp_test.to_numpy()
Xt_test_aug = np.c_[X_t_test, np.ones((len(X_t_test), 1))]

In [87]:
y_pred, y_std_nt = gpr_pe.predict(Xt_test_aug, return_std=True)

mae  = mean_absolute_error(y_pred, y_test)
rmse = np.sqrt(mean_squared_error(y_pred, y_test))
r2   = r2_score(y_pred, y_test)

print(f"MAE : {mae:.4g}")
print(f"RMSE: {rmse:.4g}")
print(f"R^2 : {r2:.4g}")

MAE : 46.27
RMSE: 47.79
R^2 : -2327


In [90]:
e2_pred_simu, simum_std = gpr_pe.predict(com_all_aug, return_std=True)
np.save(path_root+"/Data_Extraction/Results/GP-Experiments/e2_predicted_GP_simu.npy", e2_pred_simu)
np.save(path_root+"/Data_Extraction/Results/GP-Experiments/e2_predicted_GP_simu_std.npy", simum_std)